<a href="https://colab.research.google.com/github/laboratoriodecodigos/Colab-Python/blob/main/Detector_de_Somnolencia_MediaPipe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install mediapipe

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.5/36.5 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 5.6 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0


In [2]:
!wget -O face_landmarker.task https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task

--2026-07-27 21:41:36--  https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task
Resolving storage.googleapis.com (storage.googleapis.com)... 34.144.170.27, 34.153.3.27, 34.144.171.27, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|34.144.170.27|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3758596 (3.6M) [application/octet-stream]
Saving to: ‘face_landmarker.task’

face_landmarker.tas 100%[===================>]   3.58M  --.-KB/s    in 0.01s   

2026-07-27 21:41:36 (242 MB/s) - ‘face_landmarker.task’ saved [3758596/3758596]



In [1]:
from google.colab import files

uploaded = files.upload()

Saving videoplayback.mp4 to videoplayback (2).mp4


In [7]:
import os

print(os.getcwd())
print(os.listdir())

/content
['.config', 'videoplayback.mp4', 'face_landmarker.task', 'sample_data']


In [8]:
import cv2
import mediapipe as mp
import numpy as np

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# ----------------------------
# Crear detector
# ----------------------------

base_options = python.BaseOptions(
    model_asset_path="face_landmarker.task"
)

options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    num_faces=1,
    output_face_blendshapes=False,
    output_facial_transformation_matrixes=False
)

detector = vision.FaceLandmarker.create_from_options(options)

# ----------------------------
# Ojos
# ----------------------------

LEFT_EYE = [33,160,158,133,153,144]
RIGHT_EYE = [362,385,387,263,373,380]

# ----------------------------
# EAR
# ----------------------------

def EAR(points):

    A=np.linalg.norm(points[1]-points[5])
    B=np.linalg.norm(points[2]-points[4])
    C=np.linalg.norm(points[0]-points[3])

    return (A+B)/(2*C)

# ----------------------------
# Video
# ----------------------------

video="/content/videoplayback.mp4"

cap = cv2.VideoCapture(video)

if not cap.isOpened():
    print("Error: No se pudo abrir el video")
else:
    print("Video abierto correctamente")

w=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps=cap.get(cv2.CAP_PROP_FPS)

out=cv2.VideoWriter(
    "resultado.mp4",
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps,
    (w,h)
)

contador=0

while True:

    ret,frame=cap.read()

    if not ret:
        break

    rgb=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)

    mp_image=mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb
    )

    detection=detector.detect(mp_image)

    estado="DESPIERTO"

    if len(detection.face_landmarks)>0:

        face=detection.face_landmarks[0]

        puntos=[]

        for p in face:

            puntos.append(
                np.array([
                    p.x*w,
                    p.y*h
                ])
            )

        left=np.array([puntos[i] for i in LEFT_EYE])
        right=np.array([puntos[i] for i in RIGHT_EYE])

        ear=(EAR(left)+EAR(right))/2

        for p in left:
            cv2.circle(frame,(int(p[0]),int(p[1])),2,(0,255,0),-1)

        for p in right:
            cv2.circle(frame,(int(p[0]),int(p[1])),2,(0,255,0),-1)

        cv2.putText(frame,
                    f"EAR:{ear:.2f}",
                    (20,40),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1,
                    (255,255,0),
                    2)

        if ear<0.22:

            contador+=1

            if contador>15:

                estado="SOMNOLIENTO"

                cv2.putText(frame,
                            "ALERTA",
                            (20,90),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            1,
                            (0,0,255),
                            3)

        else:

            contador=0

    cv2.putText(frame,
                estado,
                (20,140),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (0,255,0) if estado=="DESPIERTO" else (0,0,255),
                3)

    out.write(frame)

cap.release()
out.release()

print("Video terminado.")

Video abierto correctamente
Video terminado.
